# Multi-Factor Technical Ensemble (MFTE) Strategy Analysis

This notebook provides a comprehensive analysis of the Ensemble Strategy (v1) for stock return prediction.

**Strategy Overview:**
- **Trend**: MACD (12, 26, 9) for directional momentum
- **Reversion**: RSI (14) for overbought/oversold conditions
- **Volatility**: Bollinger Bands (20, 2) for price channel analysis
- **Momentum**: short-term returns (5-day)
- **Volume**: Confirmation scaling

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
import os

# Add parent directory to path to allow importing strategies
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import strategy module
from strategies.strategy_ensemble_v1 import (
    calculate_features,
    predict_returns,
    get_strategy_summary
)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Configuration

In [ ]:
# Configuration
STOCK_NAME = 's1'  # Change this to analyze different stocks (s1 to s30)
DATA_DIR = '../sample_data'

print(f"Analyzing Stock: {STOCK_NAME}")
print(f"Data Directory: {DATA_DIR}")

## 2. Load Data

In [ ]:
# Load stock data
filepath = f"{DATA_DIR}/{STOCK_NAME}.npy"
A = np.load(filepath, allow_pickle=True)

# Extract columns
dates = A[:, 0]
P = A[:, 2].astype(float)  # Close prices
V = A[:, 6].astype(float)  # Volume

print(f"Loaded {len(P)} trading days")
print(f"Price range: {P.min():.2f} - {P.max():.2f}")
print(f"Volume range: {V.min():.0f} - {V.max():.0f}")

## 3. Calculate Features and Summary

In [ ]:
# Calculate technical features
df = calculate_features(P, V)

# Add dates to dataframe
df['date'] = pd.to_datetime(dates.astype(int).astype(str), format='%Y%m%d', errors='coerce')

# Get summary statistics
summary = get_strategy_summary(df)

print("="*70)
print(f"STRATEGY SUMMARY: {STOCK_NAME}")
print("="*70)
print(f"Total Trading Days: {summary['total_days']}")
print(f"Mean RSI:           {summary['mean_rsi']:.2f}")
print(f"Overbought Days:    {summary['overbought_days']}")
print(f"Oversold Days:      {summary['oversold_days']}")
print(f"Uptrend Days:       {summary['uptrend_days']}")
print(f"Downtrend Days:     {summary['downtrend_days']}")

## 4. Visualization: Price and Bollinger Bands

In [ ]:
# Create interactive plot for Bollinger Bands
fig = go.Figure()

# Bollinger Bands
fig.add_trace(go.Scatter(x=df['date'], y=df['BB_Upper'], line=dict(color='rgba(173, 216, 230, 0.4)'), name='Upper Band'))
fig.add_trace(go.Scatter(x=df['date'], y=df['BB_Lower'], line=dict(color='rgba(173, 216, 230, 0.4)'), fill='tonexty', name='Lower Band'))

fig.add_trace(go.Scatter(x=df['date'], y=df['SMA20'], line=dict(color='orange', width=1.5), name='SMA 20'))
fig.add_trace(go.Scatter(x=df['date'], y=df['price'], line=dict(color='blue', width=2), name='Close Price'))

fig.update_layout(
    title=f'Stock {STOCK_NAME} - Price and Bollinger Bands',
    xaxis_title='Date',
    yaxis_title='Price',
    height=600,
    template='plotly_white'
)

fig.show()

## 5. Visualization: MACD and RSI

In [ ]:
# Create subplots for MACD and RSI
fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, 
    vertical_spacing=0.1, 
    subplot_titles=('MACD', 'RSI')
)

# MACD
fig.add_trace(go.Scatter(x=df['date'], y=df['MACD'], line=dict(color='blue'), name='MACD'), row=1, col=1)
fig.add_trace(go.Scatter(x=df['date'], y=df['MACD_Signal'], line=dict(color='orange'), name='Signal'), row=1, col=1)
fig.add_trace(go.Bar(x=df['date'], y=df['MACD_Hist'], name='Histogram', marker_color='gray'), row=1, col=1)

# RSI
fig.add_trace(go.Scatter(x=df['date'], y=df['RSI'], line=dict(color='purple'), name='RSI'), row=2, col=1)
fig.add_trace(go.Scatter(x=df['date'], y=[70]*len(df), line=dict(color='red', dash='dash'), name='Overbought (70)'), row=2, col=1)
fig.add_trace(go.Scatter(x=df['date'], y=[30]*len(df), line=dict(color='green', dash='dash'), name='Oversold (30)'), row=2, col=1)

fig.update_layout(
    title=f'Technical Indicators for {STOCK_NAME}',
    height=700,
    template='plotly_white',
    showlegend=True
)

fig.show()

## 6. Performance Evaluation

In [ ]:
# Generate predictions
predictions = predict_returns(P, V)

# Calculate actual returns
def calculate_actual_returns(P):
    n, Q = len(P), [0]
    for i in range(1, n):
        Q.append(P[i] / P[i - 1] - 1)
    return Q

actual_returns = calculate_actual_returns(P)

# Evaluation function (matching main.py logic)
def evaluate(p, t):
    p_trimmed, t_trimmed = p[1:], t[1:]
    n = len(t_trimmed)
    e, f = [], []
    
    for i in range(1, n):
        e.append(t_trimmed[i] - p_trimmed[i - 1])
        f.append(t_trimmed[i])
    
    e = np.array(e)
    f = np.array(f)
    
    den = np.nanquantile(np.abs(e), 0.5) + 0.5 * np.nanquantile(np.abs(e), 0.9)
    num = np.nanquantile(np.abs(f), 0.5) + 0.5 * np.nanquantile(np.abs(f), 0.9)
    
    return den, 1 - den / num

# Run evaluation
abs_error, rel_score = evaluate(predictions, actual_returns)

print("="*70)
print(f"PERFORMANCE RESULTS: {STOCK_NAME}")
print("="*70)
print(f"Absolute Error: {abs_error:.6f} ({'PASS' if abs_error < 0.005 else 'FAIL'})")
print(f"Relative Score: {rel_score:.6f} ({'PASS' if rel_score > 0 else 'FAIL'})")

## 7. Comparison: Actual vs Predicted Returns

In [ ]:
comparison_df = pd.DataFrame({
    'date': df['date'],
    'actual': actual_returns,
    'predicted': predictions
})

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=comparison_df['date'], y=comparison_df['actual'], 
    name='Actual Returns', line=dict(color='blue', width=1)
))

fig.add_trace(go.Scatter(
    x=comparison_df['date'], y=comparison_df['predicted'], 
    name='Predicted Returns', line=dict(color='red', width=1)
))

fig.update_layout(
    title=f'Returns Comparison: {STOCK_NAME}',
    xaxis_title='Date',
    yaxis_title='Return',
    height=500,
    template='plotly_white',
    hovermode='x unified'
)

fig.show()